In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, chi2_contingency
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import os


In [2]:
os.getcwd()

'/home/david/Desktop/credit_analysis'

In [3]:
df_current = pd.read_csv("data/raw/current_app.csv")

In [4]:
df_current.shape

(307511, 122)

In [5]:
# converting all numerical data into 32-bit t0 reduce memory usage 

# Integer columns
int_columns = df_current.select_dtypes(include='int').columns.tolist()
# Float columns
float_columns = df_current.select_dtypes(include='float').columns.tolist()

# Convert integer columns to int32
df_current[int_columns] = df_current[int_columns].astype("int32")

# Convert float columns to float32
df_current[float_columns] = df_current[float_columns].astype("float32")


In [6]:
df_current.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
df_current.shape

(307511, 122)

In [8]:
df_current.duplicated().sum()

np.int64(0)

In [9]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [10]:
df_current.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [11]:
# transforming some columns values to positive values suitable for ML and analysis
df_current["DAYS_BIRTH"] = df_current["DAYS_BIRTH"].abs()
df_current["DAYS_EMPLOYED"] = df_current["DAYS_EMPLOYED"].replace(365243, np.nan).abs()
df_current["DAYS_REGISTRATION"] = df_current["DAYS_REGISTRATION"].abs()
df_current["DAYS_ID_PUBLISH"] = df_current["DAYS_ID_PUBLISH"].abs()
df_current["DAYS_BIRTH"] = df_current["DAYS_BIRTH"].abs()
df_current["DAYS_LAST_PHONE_CHANGE"] = df_current["DAYS_LAST_PHONE_CHANGE"].abs()


In [12]:
missing_pct = df_current.isna().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

In [13]:
# Numeric columns
df_nums = df_current.select_dtypes(include="number")

# Object/categorical columns
df_cat = df_current.select_dtypes(exclude="number")


In [14]:
COLUMNS=[col for col in df_nums.columns if col.endswith(("MEDI", "MODE","AVG"))]
corr_matrix =df_nums[COLUMNS].corr()
corr_matrix
"""
There is a very strong correlation among the mean, median, 
and mode features of the same variable, as shown by the correlation matrix.
Therefore, I will drop the mean and mode features and retain only the median feature.
"""
corr_matrix.head()

,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,TOTALAREA_MODE
APARTMENTS_AVG,1.000000,0.679389,0.100098,0.340784,0.538900,0.836958,0.611241,0.618746,0.444718,0.496852,0.943952,0.913619,0.194114,0.299719,0.973259,0.661479,0.095131,0.339222,0.529509,0.822553,0.583158,0.614498,0.436553,0.487651,0.930554,0.893463,0.187096,0.284942,0.995081,0.678977,0.099620,0.339670,0.539610,0.835123,0.607629,0.616581,0.442894,0.499052,0.941907,0.912330,0.192635,0.297454,0.892627
BASEMENTAREA_AVG,0.679389,1.000000,0.085950,0.248610,0.405459,0.564907,0.651160,0.329843,0.220908,0.467661,0.646774,0.692891,0.095418,0.266349,0.666023,0.973496,0.058421,0.249285,0.402106,0.555096,0.626382,0.328758,0.218307,0.461969,0.647432,0.677159,0.094249,0.260047,0.678174,0.994317,0.077490,0.247836,0.406592,0.562347,0.647007,0.327949,0.219523,0.469994,0.648802,0.689318,0.095287,0.265382,0.672400
YEARS_BEGINEXPLUATATION_AVG,0.100098,0.085950,1.000000,0.483564,0.091483,0.078921,0.042075,0.127573,0.167152,0.073917,0.146120,0.092736,0.034979,0.009666,0.100665,0.080754,0.971893,0.477991,0.086286,0.079578,0.037566,0.127086,0.162940,0.070008,0.141269,0.089123,0.031484,0.008627,0.100132,0.085256,0.993825,0.482951,0.091667,0.078995,0.041334,0.127273,0.166756,0.074067,0.146503,0.092175,0.034079,0.008453,0.100319
YEARS_BUILD_AVG,0.340784,0.248610,0.483564,1.000000,0.232259,0.343318,0.090591,0.519200,0.359538,0.180621,0.333937,0.355666,0.070929,0.127709,0.323250,0.233864,0.289057,0.989444,0.223498,0.334068,0.074729,0.512382,0.352092,0.170190,0.326195,0.338392,0.064494,0.112257,0.339324,0.246441,0.426564,0.998495,0.232810,0.343236,0.088385,0.518200,0.358984,0.181592,0.334945,0.354689,0.069234,0.124901,0.359263
COMMONAREA_AVG,0.538900,0.405459,0.091483,0.232259,1.000000,0.522166,0.326264,0.404014,0.295657,0.257046,0.533103,0.547030,0.105582,0.227615,0.515926,0.388210,0.048141,0.229358,0.977147,0.505542,0.301915,0.398065,0.288792,0.244198,0.526574,0.524186,0.103457,0.217018,0.538120,0.404549,0.074048,0.231841,0.995978,0.520489,0.323738,0.402250,0.294866,0.258217,0.534454,0.545501,0.105081,0.227138,0.553260


In [15]:
COLUMNS_mode_avg=[col for col in df_nums.columns if col.endswith(( "MODE","AVG"))]
df_nums =df_nums.drop(COLUMNS_mode_avg, axis=1)

"""
checking further for other columns that maybe related
"""

corr_matrix= df_nums.corr()
# Unstack the matrix into pairs
corr_pairs = corr_matrix.unstack()

# Remove self-correlations
corr_pairs = corr_pairs[corr_pairs != 1]

# Keep only strong correlations
strong_corr = corr_pairs[abs(corr_pairs) > 0.8]

# Sort by strength
df_strong_corr = strong_corr.sort_values(key=abs, ascending=False).reset_index()
corr_cols=df_strong_corr[["level_0"]].iloc[::2]

corr_cols=corr_cols.values.flatten().tolist()

index= ["SK_ID_CURR"] 

# dropping all the correlated columns 
df_nums = df_nums.drop(corr_cols + index, axis=1)

df_nums.head()

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,REGION_RATING_CLIENT_W_CITY,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,OBS_30_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,1,202500.0,406597.5,24700.5,0.018801,9461,637.0,3648.0,2120,NaN,1,1,0,1,1,0,1.0,2,10,0,0,0,0,0.083037,0.262949,0.139376,0.0369,0.9722,0.6243,0.0144,0.0690,0.0833,0.1250,0.0375,0.0193,0.0000,0.00,2.0,2.0,1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,270000.0,1293502.5,35698.5,0.003541,16765,1188.0,1186.0,291,NaN,1,1,0,1,1,0,2.0,1,11,0,0,0,0,0.311267,0.622246,NaN,0.0529,0.9851,0.7987,0.0608,0.0345,0.2917,0.3333,0.0132,0.0558,0.0039,0.01,1.0,0.0,828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,67500.0,135000.0,6750.0,0.010032,19046,225.0,4260.0,2531,26.0,1,1,1,1,1,0,1.0,2,9,0,0,0,0,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,135000.0,312682.5,29686.5,0.008019,19005,3039.0,9833.0,2437,NaN,1,1,0,1,0,0,2.0,2,17,0,0,0,0,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,0,121500.0,513000.0,21865.5,0.028663,19932,3038.0,4311.0,3458,NaN,1,1,0,1,0,0,1.0,2,11,0,0,0,1,NaN,0.322738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
""" checking percentage of missingnss in numerical columns """

missing_pct = df_nums.isna().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

# missig > 13k classify as high miss
large_miss_colums=missing_pct[:-7].index

# missig < 13k classify as small miss
small_miss_colums = missing_pct[-6:].index

In [17]:
small_miss_colums

Index(['DEF_60_CNT_SOCIAL_CIRCLE', 'OBS_30_CNT_SOCIAL_CIRCLE', 'EXT_SOURCE_2',
       'AMT_ANNUITY', 'CNT_FAM_MEMBERS', 'DAYS_LAST_PHONE_CHANGE'],
      dtype='object')

In [18]:
"""test to know whether values and missingness in  numerical large_miss_columns are informative in other to know 
how to handle missingness"""

def numerical_test(df, col_list, target):
    
    list_informative_miss = []
    list_informative_value = []
    columns = []
    missing = []

    for col in col_list:

        # -----------------------------
        # Test whether missing values are informative
        # -----------------------------
        missing_flag = df[col].isna().astype(int)

        flag_table = pd.crosstab(missing_flag, df[target])

        _, p_flag, _, _ = chi2_contingency(flag_table)

        if p_flag < 0.05:
            list_informative_miss.append("Informative")
        else:
            list_informative_miss.append("Not informative")

        # -----------------------------
        # Test whether the values themselves are informative
        # -----------------------------
        df_non_missing = df[[col, target]].dropna()

        if df_non_missing[col].nunique() < 7:

            # Chi-square test for categorical numerical variables
            value_table = pd.crosstab(df_non_missing[col],
                                      df_non_missing[target])

            _, p_value, _, _ = chi2_contingency(value_table)

        else:

            # ANOVA for continuous variables
            groups = [
                group[col].values
                for _, group in df_non_missing.groupby(target)
            ]

            _, p_value = f_oneway(*groups)

        if p_value < 0.05:
            list_informative_value.append("Informative")
        else:
            list_informative_value.append("Not Informative")
   
        missing_pct = df[col].isna().mean() * 100
        missing.append(missing_pct)
        columns.append(col)

    result = pd.DataFrame({
        "column": columns,
        "informative_miss": list_informative_miss,
        "informative_value": list_informative_value,
        "missin_pct": missing
    }).sort_values(by="missin_pct", ascending=False)

    return result

result = numerical_test(
    df=df_nums,
    col_list=large_miss_colums,
    target="TARGET")
result

,column,informative_miss,informative_value,missin_pct
0,COMMONAREA_MEDI,Informative,Informative,69.872297
1,NONLIVINGAPARTMENTS_MEDI,Informative,Not Informative,69.432963
2,FLOORSMIN_MEDI,Informative,Informative,67.848630
3,YEARS_BUILD_MEDI,Informative,Informative,66.497784
4,OWN_CAR_AGE,Informative,Informative,65.990810
5,LANDAREA_MEDI,Informative,Informative,59.376738
6,BASEMENTAREA_MEDI,Informative,Informative,58.515956
7,EXT_SOURCE_1,Informative,Informative,56.381073
8,NONLIVINGAREA_MEDI,Informative,Informative,55.179164
9,ENTRANCES_MEDI,Informative,Informative,50.348768


In [19]:
"""Since all large missing-value numerical columns are informative in their 
missingness ,we will create a missing indicator column for all of them.
And those whose real value are not informative will be dropped while the informative
will be filled using MICE method."""

Not_informative_value_cols = result.loc[result["informative_value"].str.lower()=="not informative", "column"].tolist()

informative_miss_cols = result["column"].tolist()

for col in informative_miss_cols:
        # Create missing indicator
        df_nums[f"missing_{col}"] = df_nums[col].isna().astype("int32")
        
# MICE method of filling
imputer = IterativeImputer(
estimator=RandomForestRegressor(n_estimators=5, max_depth=4, random_state=42, n_jobs=-1),           
max_iter=5, random_state=42)

cols_to_impute = [col for col in df_nums.columns if col not in Not_informative_value_cols]

df_nums[cols_to_impute] = imputer.fit_transform(df_nums[cols_to_impute])

for col in cols_to_impute:
            # this is done incase binary columns return float instead of whole number
    unique_vals = df_nums[col].dropna().unique()
    if set(unique_vals).issubset({0, 1}):
        df_nums[col]= df_nums[col].round().astype(int)
  
df_nums.drop(columns = Not_informative_value_cols, inplace=True) #dropping non value informative columns

""" the small miss-value numerical columns will be filled by simple imputation"""
for col in small_miss_colums.tolist():
    df_nums[col] = df_nums[col].fillna(df_nums[col].median())  

cols_to_round = df_nums.columns[df_nums.columns.str.contains('days|age', case=False)]

df_nums[cols_to_round] = df_nums[cols_to_round].round(0)    



In [20]:
df_nums.head()

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,REGION_RATING_CLIENT_W_CITY,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAREA_MEDI,NONLIVINGAREA_MEDI,OBS_30_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_YEAR,missing_COMMONAREA_MEDI,missing_NONLIVINGAPARTMENTS_MEDI,missing_FLOORSMIN_MEDI,missing_YEARS_BUILD_MEDI,missing_OWN_CAR_AGE,missing_LANDAREA_MEDI,missing_BASEMENTAREA_MEDI,missing_EXT_SOURCE_1,missing_NONLIVINGAREA_MEDI,missing_ENTRANCES_MEDI,missing_LIVINGAREA_MEDI,missing_FLOORSMAX_MEDI,missing_YEARS_BEGINEXPLUATATION_MEDI,missing_EXT_SOURCE_3,missing_DAYS_EMPLOYED,missing_AMT_REQ_CREDIT_BUREAU_WEEK,missing_AMT_REQ_CREDIT_BUREAU_QRT,missing_AMT_REQ_CREDIT_BUREAU_HOUR,missing_AMT_REQ_CREDIT_BUREAU_MON,missing_AMT_REQ_CREDIT_BUREAU_DAY
0,1,202500.0,406597.5,24700.5,0.018801,9461.0,637.0,3648.0,2120.0,12.0,1,1,0,1,1,0,1.0,2.0,10.0,0,0,0,0,0.083037,0.262949,0.139376,0.036900,0.972200,0.624300,0.014400,0.069000,0.0833,0.125000,0.037500,0.019300,0.000000,2.0,2.0,1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00000,1.000000,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,270000.0,1293502.5,35698.5,0.003541,16765.0,1188.0,1186.0,291.0,11.0,1,1,0,1,1,0,2.0,1.0,11.0,0,0,0,0,0.311267,0.622246,0.512523,0.052900,0.985100,0.798700,0.060800,0.034500,0.2917,0.333300,0.013200,0.055800,0.010000,1.0,0.0,828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00000,0.000000,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,0,67500.0,135000.0,6750.0,0.010032,19046.0,225.0,4260.0,2531.0,26.0,1,1,1,1,1,0,1.0,2.0,9.0,0,0,0,0,0.649026,0.555912,0.729567,0.059248,0.940569,0.287285,0.023805,0.136941,0.1723,0.184594,0.047017,0.059749,0.029387,0.0,0.0,815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00000,0.000000,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0
3,0,135000.0,312682.5,29686.5,0.008019,19005.0,3039.0,9833.0,2437.0,13.0,1,1,0,1,0,0,2.0,2.0,17.0,0,0,0,0,0.670032,0.650442,0.512523,0.059248,0.966084,0.595258,0.023805,0.136941,0.1723,0.184594,0.047017,0.059749,0.029387,2.0,0.0,617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.16961,2.329119,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1
4,0,121500.0,513000.0,21865.5,0.028663,19932.0,3038.0,4311.0,3458.0,13.0,1,1,0,1,0,0,1.0,2.0,11.0,0,0,0,1,0.649026,0.322738,0.512523,0.059248,0.901830,0.287285,0.023805,0.136941,0.1723,0.184594,0.047017,0.059749,0.029387,0.0,0.0,1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00000,0.000000,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0


In [21]:
df_cat.head()

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,WEEKDAY_APPR_PROCESS_START,ORGANIZATION_TYPE,FONDKAPREMONT_MODE,HOUSETYPE_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE
0,Cash loans,M,N,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,WEDNESDAY,Business Entity Type 3,reg oper account,block of flats,"Stone, brick",No
1,Cash loans,F,N,N,Family,State servant,Higher education,Married,House / apartment,Core staff,MONDAY,School,reg oper account,block of flats,Block,No
2,Revolving loans,M,Y,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,MONDAY,Government,NaN,NaN,NaN,NaN
3,Cash loans,F,N,Y,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,Laborers,WEDNESDAY,Business Entity Type 3,NaN,NaN,NaN,NaN
4,Cash loans,M,N,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,Core staff,THURSDAY,Religion,NaN,NaN,NaN,NaN


In [22]:
""" checking percetage of missingness in categorical columns """
missing_pct = df_cat.isna().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
missing_pct

FONDKAPREMONT_MODE     68.386172
WALLSMATERIAL_MODE     50.840783
HOUSETYPE_MODE         50.176091
EMERGENCYSTATE_MODE    47.398304
OCCUPATION_TYPE        31.345545
NAME_TYPE_SUITE         0.420148
dtype: float64

In [25]:
large_miss_catcols = missing_pct.index.tolist()[:5]
small_miss_catcols = missing_pct.index.tolist()[-1:]

def categorical_test(df, col_list,target):
    
    list_informative_miss = []
    list_informative_value = []
    columns = []
    missing = []

    for col in col_list:
    
        # -----------------------------
        # Test whether missing values are informative
        # -----------------------------
        missing_flag = df[col].isna().astype(int)
        
        flag_table = pd.crosstab(missing_flag, df[target])
        
        _, p_flag, _, _ = chi2_contingency(flag_table)
        if p_flag < 0.05:
            list_informative_miss.append("Informative")
        else:
            list_informative_miss.append("Not Informative")
        # -----------------------------
        # Test whether the values themselves are informative
        # -----------------------------
        df_non_missing = df[[col, target]].dropna()
        value_table = pd.crosstab(df_non_missing[col],
                                   df_non_missing[target])
        
        _, p_value, _, _ = chi2_contingency(value_table)
        if p_value < 0.05:
            list_informative_value.append("Informative")
        else:
            list_informative_value.append("Not Informative")
        
        missing_pct = df[col].isna().mean() * 100
        missing.append(missing_pct)
        columns.append(col)
        
    result = pd.DataFrame({
        "column": columns,
        "informative_miss": list_informative_miss,
        "informative_value": list_informative_value,
        "missin_pct": missing
    }).sort_values(by="missin_pct", ascending=False)
        
    return result

result = categorical_test(
df=pd.concat([df_cat, df_current[["TARGET"]]], axis=1),
col_list=large_miss_catcols,
    target="TARGET")
result



,column,informative_miss,informative_value,missin_pct
0,FONDKAPREMONT_MODE,Informative,Informative,68.386172
1,WALLSMATERIAL_MODE,Informative,Informative,50.840783
2,HOUSETYPE_MODE,Informative,Informative,50.176091
3,EMERGENCYSTATE_MODE,Informative,Informative,47.398304
4,OCCUPATION_TYPE,Informative,Informative,31.345545


In [26]:
del missing_pct
del result
del large_miss_colums
del small_miss_colums

df_current.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float32(65), float64(1), int32(40), object(16)
memory usage: 381.8 MB


In [27]:
# """Since all large missing-value categorical columns are informative both in their 
# missingness and in their actual values,
# we will create a missing indicator column and fill
# the original columns based on the proportion of missing values in each category."""

for col in large_miss_catcols:
    # Create missing indicator
    df_cat[f"missing_{col}"] = df_cat[col].isna().astype("int32")
    
    value_counts=df_cat[col].value_counts()
    ratios = value_counts / value_counts.sum() #calculating ratio for each category

    nan_counts= df_cat[col].isna().sum()  #counting nan value for filling
    
    fill_values = np.random.choice(
        ratios.index,           # The category names
        size=nan_counts,         # How many values to generate
        p=ratios.values         # The probabilities/ratios
    )
    df_cat.loc[df_cat[col].isna(), col] = fill_values  #Fill the NaN positions with these random values

"""small missing columns will be filled by simple imputation"""
df_cat[small_miss_catcols]=df_cat[small_miss_catcols].fillna(df_cat[small_miss_catcols].mode().iloc[0])     

In [28]:
df_cat.isna().any()

NAME_CONTRACT_TYPE             False
CODE_GENDER                    False
FLAG_OWN_CAR                   False
FLAG_OWN_REALTY                False
NAME_TYPE_SUITE                False
NAME_INCOME_TYPE               False
NAME_EDUCATION_TYPE            False
NAME_FAMILY_STATUS             False
NAME_HOUSING_TYPE              False
OCCUPATION_TYPE                False
WEEKDAY_APPR_PROCESS_START     False
ORGANIZATION_TYPE              False
FONDKAPREMONT_MODE             False
HOUSETYPE_MODE                 False
WALLSMATERIAL_MODE             False
EMERGENCYSTATE_MODE            False
missing_FONDKAPREMONT_MODE     False
missing_WALLSMATERIAL_MODE     False
missing_HOUSETYPE_MODE         False
missing_EMERGENCYSTATE_MODE    False
missing_OCCUPATION_TYPE        False
dtype: bool

In [31]:
# concatenating the data frames together 
df_current = pd.concat([df_current[index], df_nums, df_cat],axis =1)

# saving the clean data frame  into precessed data folder
df_current.to_csv("~/Desktop/credit_analysis/data/processed/df_current_clean")

In [ ]:
df_cat.isna().any()

In [ ]:
df_cat[small_miss_catcols].mode()